In [2]:
import os

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.model_selection import KFold, cross_validate
from xgboost import XGBRegressor

files_in_directory = os.listdir()

possible_file_names = [
    "modelleme_oncesi_nihai_veri_seti.csv",
    "modelleme_oncesi_nihai_veri_seti_10.csv"
]

file_name = None

for name in possible_file_names:
    if name in files_in_directory:
        file_name = name
        break

if file_name is None:
    csv_files = [file for file in files_in_directory if file.endswith(".csv")]

    if len(csv_files) == 1:
        file_name = csv_files[0]
    elif len(csv_files) > 1:
        raise FileNotFoundError(
            "Birden fazla CSV dosyası var. Lütfen doğru CSV dosyasını bırakıp tekrar çalıştır."
        )
    else:
        raise FileNotFoundError(
            "Colab ortamında CSV dosyası bulunamadı. Lütfen sol taraftan veri setini tekrar yükle."
        )

df_model = pd.read_csv(file_name)

leak_cols = [
    "batch_id",
    "scope1_kgCO2e",
    "scope2_kgCO2e",
    "scope3_tasima_kgCO2e",
    "scope3_atik_kgCO2e",
    "scope3_atiksu_kgCO2e",
    "scope3_kimyasal_kgCO2e",
    "scope3_kgCO2e"
]

cols_to_drop = [col for col in leak_cols if col in df_model.columns]

if cols_to_drop:
    df_model = df_model.drop(columns=cols_to_drop)

target_col = "toplam_karbon_kgCO2e"

if target_col not in df_model.columns:
    raise ValueError(
        f"Hedef değişken olan {target_col} veri setinde bulunamadı. Lütfen CSV dosyasını kontrol et."
    )

sifir_sayisi = (df_model[target_col] < 1).sum()

if sifir_sayisi > 0:
    print(f"UYARI: Hedef değişkende {sifir_sayisi} adet 1 kgCO2e altında değer bulunuyor.")
    print("Bu durum MAPE metriğini olumsuz etkileyebilir; MAPE sonuçları dikkatli yorumlanmalıdır.")

X = df_model.drop(columns=[target_col])
y = df_model[target_col]

bool_cols = X.select_dtypes(include=["bool"]).columns.tolist()

if bool_cols:
    X[bool_cols] = X[bool_cols].astype(int)

object_cols = X.select_dtypes(include=["object"]).columns.tolist()

for col in object_cols:
    unique_vals = set(
        X[col]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
    )

    if unique_vals.issubset({"true", "false"}):
        X[col] = (
            X[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({"true": 1, "false": 0})
        )
    elif unique_vals.issubset({"0", "1"}):
        X[col] = pd.to_numeric(X[col], errors="coerce")

non_numeric_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

if len(non_numeric_cols) > 0:
    raise ValueError(
        f"Modelleme öncesinde tüm bağımsız değişkenlerin sayısal olması gerekir. Sayısal olmayan sütunlar: {non_numeric_cols}"
    )

modeling_df_check = pd.concat([X, y], axis=1)
missing_count = modeling_df_check.isnull().sum().sum()

if missing_count > 0:
    print(modeling_df_check.isnull().sum()[modeling_df_check.isnull().sum() > 0])
    raise ValueError("Eksik değerler giderilmeden modelleme yapılmamalıdır.")

cv = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)


def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "RMSE": make_scorer(rmse_score, greater_is_better=False),
    "R2": "r2",
    "MAPE": "neg_mean_absolute_percentage_error"
}

rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

xgb_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.90,
    colsample_bytree=0.90,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,
    tree_method="hist"
)

models = {
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

results = []

for model_name, model in models.items():
    cv_results = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    mae_scores = -cv_results["test_MAE"]
    mse_scores = -cv_results["test_MSE"]
    rmse_scores = -cv_results["test_RMSE"]
    r2_scores = cv_results["test_R2"]
    mape_scores = -cv_results["test_MAPE"]

    results.append({
        "Model": model_name,
        "MAE Ortalama": mae_scores.mean(),
        "MAE Std": mae_scores.std(),
        "MSE Ortalama": mse_scores.mean(),
        "MSE Std": mse_scores.std(),
        "RMSE Ortalama": rmse_scores.mean(),
        "RMSE Std": rmse_scores.std(),
        "R² Ortalama": r2_scores.mean(),
        "R² Std": r2_scores.std(),
        "MAPE Ortalama (%)": mape_scores.mean() * 100,
        "MAPE Std (%)": mape_scores.std() * 100
    })

model_results = pd.DataFrame(results)

report_table = model_results[[
    "Model",
    "MAE Ortalama",
    "RMSE Ortalama",
    "R² Ortalama",
    "MAPE Ortalama (%)"
]].copy()

display(report_table.round(4))

,Model,MAE Ortalama,RMSE Ortalama,R² Ortalama,MAPE Ortalama (%)
0,Random Forest,228.4176,377.1147,0.9785,8.2893
1,XGBoost,177.4041,300.0713,0.9856,6.6396
